# Q4 Sarcasm Explanation & Error Analysis 

**Mohamed Fahmi Ahmed**

1. Load the en-AU adapter (best Sarcasm-F1 in Q2.3) and run it on the en-AU test set
2. Pick 10 misclassified examples (5 FP + 5 FN) by their dataset index
3. Pick 4 of them to put in a few-shot prompt, with explanations (2 FP + 2 FN for balance)
4. Test the remaining 6 with the prompt using LLaMA-3.2-3B-Instruct
5. Compare predictions before vs after few-shot prompting

## 1. Setup

In [17]:
import os, sys
import torch
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
)
from peft import PeftModel
from sklearn.metrics import f1_score

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

PyTorch: 2.11.0+cu130
CUDA   : True
GPU    : NVIDIA RTX A4000


In [18]:
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Authenticated")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Authenticated


In [19]:
BASE_MODEL    = "facebook/opt-1.3b"
ADAPTER_REPO  = "momofahmi/besstie-lora-en-au-opt-1.3b"
VARIETY       = "en-AU"
TASK          = "Sarcasm"
MAX_LENGTH    = 128
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Variety: {VARIETY}  ·  Adapter: {ADAPTER_REPO}")

Variety: en-AU  ·  Adapter: momofahmi/besstie-lora-en-au-opt-1.3b


## 2. Load model and run predictions on the en-AU test set

In [20]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.float16 if torch.cuda.is_available() else torch.float32
base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=2, dtype=dtype,
)
base_model.config.pad_token_id = tokenizer.pad_token_id

peft_model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
peft_model.eval()
peft_model = peft_model.to(DEVICE)
print("Adapter loaded")

Loading weights: 100%|██████████| 388/388 [00:00<00:00, 50502.42it/s]
OPTForSequenceClassification LOAD REPORT from: facebook/opt-1.3b
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Adapter loaded


In [21]:
ds = load_dataset("surrey-nlp/BESSTIE-CW-26")
test_au = ds["test"].filter(lambda x: x["variety"] == VARIETY)
print(f"en-AU test set: {len(test_au)} examples")

def predict_batch(texts, model, tokenizer, max_len=128, batch_size=16):
    preds, confs = [], []
    for i in range(0, len(texts), batch_size):
        batch = tokenizer(
            texts[i:i+batch_size],
            truncation=True, padding="max_length",
            max_length=max_len, return_tensors="pt",
        ).to(DEVICE)
        with torch.no_grad():
            logits = model(**batch).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        preds.extend(probs.argmax(axis=-1).tolist())
        confs.extend(probs.max(axis=-1).tolist())
    return np.array(preds), np.array(confs)

texts  = list(test_au["text"])
y_true = np.array([int(x) for x in test_au[TASK]])
y_pred, y_conf = predict_batch(texts, peft_model, tokenizer, MAX_LENGTH)

macro_f1   = f1_score(y_true, y_pred, average="macro")
sarcasm_f1 = f1_score(y_true, y_pred, average=None)[1]
print(f"Macro-F1  : {macro_f1:.4f}")
print(f"Sarcasm-F1: {sarcasm_f1:.4f}  (expected ~0.68 from Q2.3)")

en-AU test set: 667 examples
Macro-F1  : 0.7771
Sarcasm-F1: 0.7002  (expected ~0.68 from Q2.3)


## 3. Build the errors dataframe

We index every error by its position in the test set (the `idx` column).
This idx will be the only handle we use afterwards every selection,
every filter, every join goes through `idx`.

In [22]:
errors = []
for i, (text, true, pred, conf) in enumerate(zip(texts, y_true, y_pred, y_conf)):
    if true != pred:
        errors.append({
            "idx": i,
            "text": text,
            "true_label": "Sarcastic" if true == 1 else "Not Sarcastic",
            "predicted":  "Sarcastic" if pred == 1 else "Not Sarcastic",
            "error_type": "False Positive" if pred == 1 else "False Negative",
            "confidence": round(float(conf), 3),
            "text_length": len(text.split()),
        })

errors_df = pd.DataFrame(errors)
n_fp = (errors_df["error_type"] == "False Positive").sum()
n_fn = (errors_df["error_type"] == "False Negative").sum()
print(f"Total errors: {len(errors_df)}  ·  FP: {n_fp}  ·  FN: {n_fn}")

Total errors: 131  ·  FP: 88  ·  FN: 43


In [23]:
fp_top = errors_df[errors_df["error_type"] == "False Positive"]\
    .sort_values("confidence", ascending=False).head(10).reset_index(drop=True)
fn_top = errors_df[errors_df["error_type"] == "False Negative"]\
    .sort_values("confidence", ascending=False).head(10).reset_index(drop=True)

print("─── Top 10 False Positives (model said Sarcastic, but it's not) ───")
for _, r in fp_top.iterrows():
    print(f"  idx={r['idx']:4d}  conf={r['confidence']:.2f}  | {r['text'][:140]}")

print("\n─── Top 10 False Negatives (model said Not Sarcastic, but it is) ───")
for _, r in fn_top.iterrows():
    print(f"  idx={r['idx']:4d}  conf={r['confidence']:.2f}  | {r['text'][:140]}")

─── Top 10 False Positives (model said Sarcastic, but it's not) ───
  idx= 492  conf=1.00  | By never telling us where she's from!
  idx= 256  conf=1.00  | For once, I kind of agree with the North Shore NIMBYs. Sydney has enough people. We're already packed like sardines, schools are overflowing
  idx= 508  conf=1.00  | Not a shill mate. Just someone that's pissed off his rent has increased by 50% since 2020.
  idx= 395  conf=1.00  | When will the internet stop being racist? Is that really your question?
  idx= 320  conf=0.99  | Aren't they supposed to talk? 
And that Philippine president is just looking for an excuse to declare martial law and stay in power for two 
  idx= 596  conf=0.99  | When I ask for extra sauce, I only want one extra squirt out of the bottle, not the entire bottle. Stop hiring restarted people.
  idx= 458  conf=0.99  | The first two words in the title are unnecessary...
  idx= 618  conf=0.99  | 5 weeks of annual leave is standard for shift workers.
The other 4 w

## 4. Select exactly 10 errors by index

We pick 5 False Positives + 5 False Negatives from the top-confidence
lists. 

In [35]:
SELECTED_FP_IDX = [492, 256, 508, 395, 618]    
SELECTED_FN_IDX = [264, 302, 657, 142, 523]   

SELECTED_IDX = SELECTED_FP_IDX + SELECTED_FN_IDX

assert len(SELECTED_IDX) == 10, f"Expected 10 idx, got {len(SELECTED_IDX)}"
assert len(set(SELECTED_IDX)) == 10, "Duplicate idx in selection"

print(f"Selected 10 errors by idx (5 FP + 5 FN):")
print(f"  FP idx: {SELECTED_FP_IDX}")
print(f"  FN idx: {SELECTED_FN_IDX}")

Selected 10 errors by idx (5 FP + 5 FN):
  FP idx: [492, 256, 508, 395, 618]
  FN idx: [264, 302, 657, 142, 523]


In [36]:
selected = errors_df[errors_df["idx"].isin(SELECTED_IDX)].copy()
selected = selected.sort_values("error_type").reset_index(drop=True)

assert len(selected) == 10, f"Filter dropped rows; expected 10 got {len(selected)}"

for _, row in selected.iterrows():
    print(f"\n[idx={row['idx']}] {row['error_type']}  conf={row['confidence']:.2f}")
    print(f"   True: {row['true_label']}  ·  Predicted: {row['predicted']}")
    print(f"   Text: {row['text'][:200]}")


[idx=142] False Negative  conf=0.97
   True: Sarcastic  ·  Predicted: Not Sarcastic
   Text: It's great, barely any customers and the cinemas are always empty.

[idx=264] False Negative  conf=1.00
   True: Sarcastic  ·  Predicted: Not Sarcastic
   Text: If the pain is unbearable you need to go to a hospital, it will be free

[idx=302] False Negative  conf=1.00
   True: Sarcastic  ·  Predicted: Not Sarcastic
   Text: We recently spent a fair bit of money on a very nice dining table and sturdy comfortable dining chairs... So to answer your question we eat on the lounge watching our stories.

[idx=523] False Negative  conf=1.00
   True: Sarcastic  ·  Predicted: Not Sarcastic
   Text: The line is incredibly long. Expect to wait an hour if you go after 10 pm. Also they cutoff their custom cocktails 40-50 minutes and sometimes an hour before they close at 1 am, so basically when they

[idx=657] False Negative  conf=0.99
   True: Sarcastic  ·  Predicted: Not Sarcastic
   Text: I was meaning 

In [37]:
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)
selected.to_csv(os.path.join(RESULTS_DIR, "q4_errors_10.csv"), index=False)
print(f"Saved: results/q4_errors_10.csv")

Saved: results/q4_errors_10.csv


## 5. Choose 4 examples for the few-shot prompt

We pick 2 FP + 2 FN for balance. Selecting by idx means the 6 remaining
will automatically be the right ones.

In [38]:
PROMPT_EXPLANATIONS = {
    142: (
        "This text is sarcastic. The speaker says 'It's great' but immediately "
        "describes facts that contradict the positive evaluation: 'barely any "
        "customers' and 'always empty' are bad outcomes for a cinema. The "
        "explicit positive/negative reversal in the same sentence is the cue."
    ),
    302: (
        "This text is sarcastic. The structure 'we bought X, so we do Y' where "
        "Y contradicts X (paying for nice dining furniture but eating on the "
        "lounge anyway) is a classic self-mocking sarcastic confession. The "
        "phrase 'so to answer your question' sets up the punchline."
    ),
    508: (
        "This text is not sarcastic. The speaker explicitly clarifies their "
        "position with 'Not a shill mate' and then describes a real grievance "
        "('rent has increased by 50% since 2020'). The emotion ('pissed off') "
        "is a sincere reaction, not an ironic mask."
    ),
    618: (
        "This text is not sarcastic. It is a factual statement about Australian "
        "shift-worker leave entitlements. There is no positive/negative reversal "
        "and no exaggeration; the numbers and acronyms (ADOs, 38 hour week) are "
        "informative, not ironic."
    ),
}

PROMPT_IDX = list(PROMPT_EXPLANATIONS.keys())
assert len(PROMPT_IDX) == 4, f"Expected 4 prompt idx, got {len(PROMPT_IDX)}"
assert all(idx in SELECTED_IDX for idx in PROMPT_IDX), \
    "Some PROMPT_IDX are not in SELECTED_IDX — must select from the 10"
assert len(set(PROMPT_IDX)) == 4, "Duplicate idx in PROMPT_IDX"

print(f"Prompt will use these 4 idx: {PROMPT_IDX}")
print(f"Test set will use these 6 idx: {[i for i in SELECTED_IDX if i not in PROMPT_IDX]}")

Prompt will use these 4 idx: [142, 302, 508, 618]
Test set will use these 6 idx: [492, 256, 395, 264, 657, 523]


In [39]:
print(f"SELECTED_IDX: {sorted(SELECTED_IDX)}")
print(f"PROMPT_IDX:   {PROMPT_IDX}")
print(f"Missing:      {[i for i in PROMPT_IDX if i not in SELECTED_IDX]}")

SELECTED_IDX: [142, 256, 264, 302, 395, 492, 508, 523, 618, 657]
PROMPT_IDX:   [142, 302, 508, 618]
Missing:      []


In [40]:
def get_row_by_idx(idx):
    matches = selected[selected["idx"] == idx]
    assert len(matches) == 1, f"idx {idx} not found uniquely in selected"
    return matches.iloc[0]

few_shot_examples = []
for idx, explanation in PROMPT_EXPLANATIONS.items():
    row = get_row_by_idx(idx)
    few_shot_examples.append({
        "idx":         int(row["idx"]),
        "text":        row["text"],
        "true_label":  row["true_label"],
        "error_type":  row["error_type"],
        "explanation": explanation,
    })

print(f"Built {len(few_shot_examples)} few-shot examples:")
for ex in few_shot_examples:
    print(f"\n  idx={ex['idx']}  ({ex['error_type']}, true={ex['true_label']})")
    print(f"  Text: {ex['text'][:150]}")

Built 4 few-shot examples:

  idx=142  (False Negative, true=Sarcastic)
  Text: It's great, barely any customers and the cinemas are always empty.

  idx=302  (False Negative, true=Sarcastic)
  Text: We recently spent a fair bit of money on a very nice dining table and sturdy comfortable dining chairs... So to answer your question we eat on the lou

  idx=508  (False Positive, true=Not Sarcastic)
  Text: Not a shill mate. Just someone that's pissed off his rent has increased by 50% since 2020.

  idx=618  (False Positive, true=Not Sarcastic)
  Text: 5 weeks of annual leave is standard for shift workers.
The other 4 weeks are usually ADOs from working shifts above the standard 38 hour week.
It's a 


In [41]:
remaining = selected[~selected["idx"].isin(PROMPT_IDX)].reset_index(drop=True)

assert len(remaining) == 6, f"Expected 6 remaining errors, got {len(remaining)}"
print(f"6 errors remain for testing (not shown to the model in the prompt):")
for _, r in remaining.iterrows():
    print(f"\n  idx={r['idx']}  ({r['error_type']}, true={r['true_label']})")
    print(f"  Text: {r['text'][:150]}")

6 errors remain for testing (not shown to the model in the prompt):

  idx=264  (False Negative, true=Sarcastic)
  Text: If the pain is unbearable you need to go to a hospital, it will be free

  idx=523  (False Negative, true=Sarcastic)
  Text: The line is incredibly long. Expect to wait an hour if you go after 10 pm. Also they cutoff their custom cocktails 40-50 minutes and sometimes an hour

  idx=657  (False Negative, true=Sarcastic)
  Text: I was meaning there isn't a medication for autism that will'fix' it, compared to adhd.
Although, some antidepressants or diazepam meds help with anxie

  idx=256  (False Positive, true=Not Sarcastic)
  Text: For once, I kind of agree with the North Shore NIMBYs. Sydney has enough people. We're already packed like sardines, schools are overflowing and traff

  idx=395  (False Positive, true=Not Sarcastic)
  Text: When will the internet stop being racist? Is that really your question?

  idx=492  (False Positive, true=Not Sarcastic)
  Text: By ne

## 6. Build the few-shot prompt template

In [42]:
def build_prompt(examples, query_text):
    """
    Few-shot prompt format:
      - System instruction
      - 4 worked examples with label + explanation
      - The query text with empty Yes/No slot
    """
    parts = []
    parts.append(
        "You are a sarcasm detector for English text. "
        "Sarcasm is saying the opposite of what is meant, "
        "often to mock, criticise, or express frustration. "
        "Read each text carefully and decide whether it is sarcastic.\n"
    )
    parts.append("Here are four examples with explanations:\n")
    for i, ex in enumerate(examples, 1):
        parts.append(f"Example {i}:")
        parts.append(f'Text: "{ex["text"]}"')
        parts.append(f'Is this sarcastic? {"Yes" if ex["true_label"] == "Sarcastic" else "No"}')
        parts.append(f'Explanation: {ex["explanation"]}\n')
    parts.append("Now classify this new text:")
    parts.append(f'Text: "{query_text}"')
    parts.append("Is this sarcastic?")
    return "\n".join(parts)


sample_query = remaining.iloc[0]["text"]
sample_prompt = build_prompt(few_shot_examples, sample_query)
print(sample_prompt)

You are a sarcasm detector for English text. Sarcasm is saying the opposite of what is meant, often to mock, criticise, or express frustration. Read each text carefully and decide whether it is sarcastic.

Here are four examples with explanations:

Example 1:
Text: "It's great, barely any customers and the cinemas are always empty."
Is this sarcastic? Yes
Explanation: This text is sarcastic. The speaker says 'It's great' but immediately describes facts that contradict the positive evaluation: 'barely any customers' and 'always empty' are bad outcomes for a cinema. The explicit positive/negative reversal in the same sentence is the cue.

Example 2:
Text: "We recently spent a fair bit of money on a very nice dining table and sturdy comfortable dining chairs... So to answer your question we eat on the lounge watching our stories."
Is this sarcastic? Yes
Explanation: This text is sarcastic. The structure 'we bought X, so we do Y' where Y contradicts X (paying for nice dining furniture but 

## 7. Load LLaMA-3.2-3B-Instruct

We use an instruction-tuned model rather than OPT-1.3B because OPT is not
trained to follow instructions.

In [44]:
LLAMA_MODEL = "meta-llama/Llama-3.2-1B-Instruct"

print(f"Loading {LLAMA_MODEL}...")
llama_tok = AutoTokenizer.from_pretrained(LLAMA_MODEL)
if llama_tok.pad_token is None:
    llama_tok.pad_token = llama_tok.eos_token

llama_model = AutoModelForCausalLM.from_pretrained(
    LLAMA_MODEL,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
llama_model.eval()
print("LLaMA Instruct loaded")

Loading meta-llama/Llama-3.2-1B-Instruct...


Loading weights: 100%|██████████| 146/146 [00:21<00:00,  6.84it/s]


LLaMA Instruct loaded


In [45]:
def llama_generate(prompt, max_new_tokens=20):
    """Greedy generation — deterministic."""
    inputs = llama_tok(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        output = llama_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=llama_tok.eos_token_id,
        )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return llama_tok.decode(new_tokens, skip_special_tokens=True).strip()


def parse_yes_no(generated_text):
    """Extract a Yes/No answer. Returns 'Sarcastic', 'Not Sarcastic', or 'UNKNOWN'."""
    text = generated_text.lower().strip()
    if text.startswith("yes"): return "Sarcastic"
    if text.startswith("no"):  return "Not Sarcastic"
    if "yes" in text and "no" not in text: return "Sarcastic"
    if "no" in text and "yes" not in text: return "Not Sarcastic"
    return "UNKNOWN"


test_resp = llama_generate(sample_prompt)
print(f"Raw  : {test_resp!r}")
print(f"Parse: {parse_yes_no(test_resp)}")

The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Raw  : "Yes\nExplanation: This text is sarcastic. The speaker says 'if the pain is unbearable'"
Parse: Sarcastic


## 8. Run few-shot inference on the 6 remaining examples

In [46]:
n = len(remaining)
results = []

for i, row in remaining.iterrows():
    text = row["text"]
    prompt = build_prompt(few_shot_examples, text)
    raw = llama_generate(prompt)
    pred = parse_yes_no(raw)
    corrected = (pred == row["true_label"])

    results.append({
        "idx":              int(row["idx"]),
        "text_preview":     text[:80] + ("..." if len(text) > 80 else ""),
        "true_label":       row["true_label"],
        "OPT_prediction":   row["predicted"],
        "LLaMA_few_shot":   pred,
        "corrected":        "yes" if corrected else "no",
        "raw_output":       raw[:80],
    })
    print(f"[{i+1}/{n}]  LLaMA: {raw[:60]:60s}  ->  {pred}  ({'CORRECT' if corrected else 'still wrong'})")

results_df = pd.DataFrame(results)

assert len(results_df) == 6, f"Expected 6 results, got {len(results_df)}"
print(f"\nDone — 6 examples tested.")

[1/6]  LLaMA: Yes
Explanation: This text is sarcastic. The speaker says 'i  ->  Sarcastic  (CORRECT)
[2/6]  LLaMA: Yes
Explanation: This text is sarcastic. The speaker describ  ->  Sarcastic  (CORRECT)
[3/6]  LLaMA: Yes
Explanation: This text is sarcastic. The speaker says 'I  ->  Sarcastic  (CORRECT)
[4/6]  LLaMA: Yes
Explanation: This text is sarcastic. The speaker says 'I  ->  Sarcastic  (still wrong)
[5/6]  LLaMA: Yes
Explanation: This text is sarcastic. The speaker says 'W  ->  Sarcastic  (still wrong)
[6/6]  LLaMA: Yes
Explanation: This text is sarcastic. The speaker says 'n  ->  Sarcastic  (still wrong)

Done — 6 examples tested.


In [48]:
print("BEFORE vs AFTER  few-shot prompting on the 6 remaining errors")
display_cols = ["idx", "text_preview", "true_label", "OPT_prediction",
                "LLaMA_few_shot", "corrected"]
print(results_df[display_cols].to_string(index=False))

n_corrected = (results_df["corrected"] == "yes").sum()
n_unknown   = (results_df["LLaMA_few_shot"] == "UNKNOWN").sum()
n_total     = len(results_df)

print(f"\nCorrected by few-shot: {n_corrected}/{n_total}")
print(f"Still wrong          : {n_total - n_corrected - n_unknown}/{n_total}")
print(f"Unparseable LLaMA out: {n_unknown}/{n_total}")

BEFORE vs AFTER  few-shot prompting on the 6 remaining errors
 idx                                                                        text_preview    true_label OPT_prediction LLaMA_few_shot corrected
 264             If the pain is unbearable you need to go to a hospital, it will be free     Sarcastic  Not Sarcastic      Sarcastic       yes
 523 The line is incredibly long. Expect to wait an hour if you go after 10 pm. Also ...     Sarcastic  Not Sarcastic      Sarcastic       yes
 657 I was meaning there isn't a medication for autism that will'fix' it, compared to...     Sarcastic  Not Sarcastic      Sarcastic       yes
 256 For once, I kind of agree with the North Shore NIMBYs. Sydney has enough people.... Not Sarcastic      Sarcastic      Sarcastic        no
 395             When will the internet stop being racist? Is that really your question? Not Sarcastic      Sarcastic      Sarcastic        no
 492                                               By never telling us where she

In [49]:
fp_results = results_df.merge(
    selected[["idx", "error_type"]], on="idx", how="left"
)
print("Per error type:")
for et in ["False Positive", "False Negative"]:
    sub = fp_results[fp_results["error_type"] == et]
    if len(sub) == 0: continue
    n_corr = (sub["corrected"] == "yes").sum()
    print(f"  {et:18s}: {n_corr}/{len(sub)} corrected")

Per error type:
  False Positive    : 0/3 corrected
  False Negative    : 3/3 corrected


In [50]:
results_df.to_csv(os.path.join(RESULTS_DIR, "q4_few_shot_results.csv"), index=False)
print(f"Saved: results/q4_few_shot_results.csv")

with open(os.path.join(RESULTS_DIR, "q4_prompt_template.txt"), "w") as f:
    f.write(build_prompt(few_shot_examples, "<text to classify>"))
print("Saved: results/q4_prompt_template.txt")

Saved: results/q4_few_shot_results.csv
Saved: results/q4_prompt_template.txt
